In [4]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark started successfully!")

C:\Users\harsh\PycharmProjects\JupyterProject1\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Version: 4.2.0
Spark started successfully!


 Q1: Roles of Driver, Cluster Manager, and Executor in Spark

Driver:
- The main program that runs the user's code
- Creates the DAG (execution plan)
- Distributes tasks to Executors
- Collects the final result

Cluster Manager:
- Manages resources (CPU, RAM) across the cluster
- Allocates resources to the Driver
- Types: YARN, Mesos, Kubernetes, Standalone

Executor:
- Performs the actual data processing
- Runs tasks in parallel
- Sends results back to the Driver

Flow:
User Code → Driver → Cluster Manager → Executors → Result

 Q2: How Spark's Lazy Evaluation Improves Performance

Lazy Evaluation means Spark does NOT execute transformations immediately when they are written. Instead, it builds an  execution plan (DAG) and only executes when an Action is called.

How it improves performance:

1. Optimization before execution:
   Spark analyzes the entire DAG before running anything and finds the most efficient execution path.

2. Filter pushdown:
   If a filter() is written after many transformations,Spark automatically moves it earlier in the plan so less data is processed overall.

3. Avoids unnecessary computation:
   If a transformation result is never used,Spark simply skips it entirely.

4. Combines multiple steps:
   Multiple transformations are combined into fewer stages to minimize data shuffling.

**Example:**
df.filter().select().groupBy() — none of these execute until .show() or .count() is called at the end.

In [5]:
# Q3: Read a CSV file with header and inferSchema enabled
df = spark.read.csv(
    r"C:\Users\harsh\Desktop\Celebal Technologies Internship\Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

Q4: Difference between CSV and Parquet

CSV (Row-based storage):
- Data is stored row by row
- To read 1 column, entire row must be read
- No compression — larger file size
- No schema stored — must be defined every time
- Example: reading Sales column reads all 21 columns

Parquet (Columnar storage):
- Data is stored column by column
- Only required columns are read — rest are skipped
- High compression — much smaller file size
- Schema is stored inside the file
- Supports Predicate Pushdown

Why does it matter for performance?

CSV: Reads all columns → More disk I/O → Slower.

Parquet: Reads only required columns → Less disk I/O → Faster and more memory-efficient.

In [5]:
# Q5: Select product_id and price where category is 'Electronics'
# Note: Superstore dataset has 'Technology' instead of 'Electronics'
# Using 'Technology' as equivalent category
# 'Product ID' = product_id, 'Sales' = price, 'Category' = category

df_result = df.filter(df['Category'] == 'Technology').select('Product ID', 'Sales')

df_result.show(5)

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
+---------------+--------+
only showing top 5 rows


In [6]:
# Q6: Rename column old_name to new_name and cast price column from String to Double
# Note: Using Superstore dataset columns:
# 'Customer Name' renamed to 'customer_name'
# 'Sales' cast from String to Double

from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

df_revised = df.withColumnRenamed('Customer Name', 'customer_name').withColumn('Sales', col('Sales').cast(DoubleType()))

print("After renaming and casting:")
df_revised.select('customer_name', 'Sales').show(5)
df_revised.printSchema()

After renaming and casting:
+---------------+--------+
|  customer_name|   Sales|
+---------------+--------+
|    Claire Gute|  261.96|
|    Claire Gute|  731.94|
|Darrin Van Huff|   14.62|
| Sean O'Donnell|957.5775|
| Sean O'Donnell|  22.368|
+---------------+--------+
only showing top 5 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- S

Q7: How Spark uses Lineage Graph (DAG) for Fault Tolerance

When a worker node fails during execution, Spark does not restart the entire job. Instead it uses the Lineage Graph (DAG)
to recover only the lost data.

How it works:

1. Spark maintains a complete record of all transformations applied to the original data — this is the Lineage Graph.

2. If a worker node fails and loses its partition of data, Spark looks at the Lineage Graph to find:
   - Where the original data came from
   - Which transformations were applied to get to that partition

3. Spark recomputes ONLY the lost partition from the last known good point — not the entire dataset.

Example:
Original CSV → filter → groupBy → [Node 3 fails here]

Spark recomputes:
Original CSV → filter → groupBy → only Node 3's partition

Why this is powerful:
- No need to save intermediate results to disk
- Only lost partitions are recomputed
- Rest of the job continues unaffected

In [10]:
# Q8: Filter orders where status is 'Completed' AND amount > 1000
#
# Note: Superstore dataset does not have 'status' and 'amount' columns
# Using 'Ship Mode' as status and 'Sales' as amount

from pyspark.sql.functions import col

df_result = df.filter(
    (df['Ship Mode'] == 'Second Class') &
    (col('Sales').try_cast('double') > 1000)
)

df_result.select('Order ID', 'Ship Mode', 'Sales').show(5)
print("Total rows:", df_result.count())

+--------------+------------+--------+
|      Order ID|   Ship Mode|   Sales|
+--------------+------------+--------+
|CA-2014-131926|Second Class| 2001.86|
|CA-2014-131926|Second Class| 1503.25|
|US-2014-106992|Second Class|3059.982|
|US-2014-106992|Second Class|2519.958|
|US-2015-161991|Second Class|  1114.4|
+--------------+------------+--------+
only showing top 5 rows
Total rows: 94


Q9: Predicate Pushdown in Parquet

Predicate Pushdown is an optimization technique where Spark pushes filter conditions directly to the data source level,so only relevant data is loaded into memory.

How it works with Parquet:

Normal flow without Predicate Pushdown:
1. Read entire Parquet file into memory (all rows)
2. Apply filter in memory
3. Return matching rows
→ Wastes memory loading unnecessary data

Flow WITH Predicate Pushdown:
1. Spark sends filter condition to Parquet reader
2. Parquet reader checks column statistics (min/max values) stored in each row group
3. Skips entire row groups that don't match the filter
4. Only matching data is loaded into memory
→ Much less data loaded = faster and memory efficient

Example:
Filter: Sales > 5000
- Without Pushdown: loads all 9994 rows, then filters
- With Pushdown: skips row groups where max Sales < 5000, loads only relevant row groups

**Why Parquet supports this better than CSV:**
Parquet stores column statistics (min, max, count) for each row group — CSV has no such metadata, so pushdown is not possible.

In [11]:
# Q10: Add a new column final_price = base_price * 1.18 (18% tax)
# Note: Using 'Sales' as base_price in Superstore dataset

from pyspark.sql.functions import col

df_result = df.withColumn(
    'final_price',
    col('Sales').try_cast('double') * 1.18
)

df_result.select('Product Name', 'Sales', 'final_price').show(5)

+--------------------+--------+------------------+
|        Product Name|   Sales|       final_price|
+--------------------+--------+------------------+
|Bush Somerset Col...|  261.96|309.11279999999994|
|Hon Deluxe Fabric...|  731.94|          863.6892|
|Self-Adhesive Add...|   14.62|           17.2516|
|Bretford CR4500 S...|957.5775|        1129.94145|
|Eldon Fold 'N Rol...|  22.368|26.394239999999996|
+--------------------+--------+------------------+
only showing top 5 rows


Q11: Difference between Transformations and Actions

Transformations:
- Create a new DataFrame from existing one
- Lazily evaluated — not executed immediately
- Just added to the DAG execution plan

Examples:
1. filter() — filters rows based on condition
2. select() — selects specific columns
3. groupBy() — groups data by column
4. withColumn() — adds or modifies a column

Actions:
- Trigger the actual execution of the DAG
- Return a result to the Driver or write to storage

Examples:
1. show() — displays rows on screen
2. count() — returns number of rows
3. collect() — returns all rows to Driver
4. write() — saves DataFrame to storage

In [1]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'

In [6]:
# Q12: Load Parquet file, filter null Customer ID, save as CSV
# Note: Using 'Customer ID' as equivalent of user_id

from pyspark.sql.functions import col

# Step 1: Save Superstore data as Parquet first
df.write.mode('overwrite').parquet('superstore.parquet')
print("Saved as Parquet!")

# Step 2: Load Parquet file
df_parquet = spark.read.parquet('superstore.parquet')
print("Loaded from Parquet:")
df_parquet.show(3)

# Step 3: Filter out rows where Customer ID is null
df_filtered = df_parquet.filter(col('Customer ID').isNotNull())
print("After filtering nulls:")
print("Total rows:", df_filtered.count())

# Step 4: Save as CSV
df_filtered.write.mode('overwrite').csv(
    'output_superstore',
    header=True
)
print("Saved as CSV successfully!")



Py4JJavaError: An error occurred while calling o37.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(Fi

In [ ]:
# Q12: Load Parquet file, filter null user_id, save as CSV
#
# Note: Parquet write/read requires Hadoop winutils setup on Windows.
# The concept is demonstrated below — in a real cloud environment
# like Databricks or EMR this would work directly without any setup.
#
# Spark command for reference:
# df_parquet = spark.read.parquet("path/to/input")
# df_filtered = df_parquet.filter(col('Customer ID').isNotNull())
# df_filtered.write.mode('overwrite').csv("path/to/output", header=True)

Q13: Client Mode vs Cluster Mode in Spark

Client Mode:
- Driver runs on the machine that submitted the job
- Driver is outside the cluster
- If client machine disconnects, job fails
- Used for: interactive jobs, debugging, development
- Example: running PySpark from your laptop

Cluster Mode:
- Driver runs inside the cluster
- Driver is managed by the Cluster Manager
- Client machine can disconnect — job continues
- Used for: production jobs, scheduled pipelines
- Example: submitting jobs on Databricks/EMR



In [7]:
# Q14: Filter dataset where region is 'North' OR priority is 'High'
#
# Note: Superstore dataset does not have 'region' and 'priority' columns
# Using 'Region' as region and 'Ship Mode' as priority equivalent
# 'First Class' used as equivalent for 'High' priority

from pyspark.sql.functions import col

df_result = df.filter(
    (df['Region'] == 'West') |
    (df['Ship Mode'] == 'First Class')
)

df_result.select('Order ID', 'Region', 'Ship Mode').show(5)
print("Total rows:", df_result.count())

+--------------+------+--------------+
|      Order ID|Region|     Ship Mode|
+--------------+------+--------------+
|CA-2016-138688|  West|  Second Class|
|CA-2014-115812|  West|Standard Class|
|CA-2014-115812|  West|Standard Class|
|CA-2014-115812|  West|Standard Class|
|CA-2014-115812|  West|Standard Class|
+--------------+------+--------------+
only showing top 5 rows
Total rows: 4226


 Q15: Why use .show(5) instead of .collect() on large datasets?

.collect():
- Brings ALL rows from all Executors to the Driver machine
- On a multi-terabyte dataset this means:
  - Billions of rows transferred over network
  - Driver runs out of RAM and crashes
  - Entire application fails
- Example: 1TB dataset → Driver needs 1TB RAM to collect!

.show(5):
- Fetches ONLY 5 rows from the cluster
- Spark stops processing as soon as 5 rows are found
- Driver memory is barely used
- Safe to use on any size dataset

